# Parked sections from castalign_testground.ipynb

Cells removed 2026-05-07 — kept here for future re-implementation. Copy a section back into `alignment/castalign_testground.ipynb` when needed.

Parked sections:
- §10. Post-Alignment Verification — warp + Pearson + 3-layer napari QC
- §11. Nonlinear Refinement (Triangulation2D / Triangulation)
- §12. Warped Image Export — multi-page TIFF in block or invivo space
- §13. Graph Reload Verification — SQLite persistence sanity check
- §17. Advanced: alignment_gui() with crop & references
- §18. Advanced: Node-Name Mode
- §19. Troubleshooting & Tips (static doc)
- §20. MNN Alignment Validation — Cellpose centroid + KDTree MNN + histogram

**Dependencies on the main notebook:** these cells expect globals from §4 (`get_selected_slice`, `slice_nodes`, `aligned_slices`), alignment globals (`block_node`, `invivo_node`, `g`), and in some cases each other (e.g., §12 warped export reuses `warped_arr` from §10). Restore them by pasting back in roughly their original order, then re-run the prerequisite cells in the main notebook first.


---

## 10. Post-Alignment Verification

Applies the current transform, computes Pearson correlation, and opens a napari viewer with three layers:
- **Fixed** (gray) â€” the reference image
- **Warped** (green) â€” the transformed movable image
- **Original** (red, hidden) â€” the untransformed movable for comparison

Set `verify_target` to choose which alignment to verify:
- `block_node` â€” direct slice â†’ block edge
- `invivo_node` â€” full two-hop path (slice â†’ block â†’ invivo, composed automatically)

In [ ]:
# =============================================================================
# POST-ALIGNMENT VERIFICATION
# =============================================================================

# --- Choose verification target ---
slice_name = get_selected_slice()

# Option 1: Verify slice -> block (direct edge)
verify_target = block_node
# Option 2: Verify slice -> invivo (full two-hop, composed automatically)
# verify_target = invivo_node

verify_fixed = g.get_image(verify_target)
verify_movable = g.get_image(slice_name)
verify_transform = g.get_transform(slice_name, verify_target)

print(f"Verifying alignment: {slice_name} -> {verify_target}")
print(f"Transform: {type(verify_transform).__name__}")
print()

# Compute quality
r, warped_arr = compute_alignment_quality(verify_fixed, verify_movable, verify_transform)
print(f"Pearson correlation (overlapping region): r = {r:.4f}")
print()

# Open napari viewer
import napari

viewer = napari.Viewer(title=f"Verification: {slice_name} -> {verify_target}")
viewer.add_image(np.asarray(verify_fixed), name="fixed", colormap="gray", blending="additive", opacity=0.7)
viewer.add_image(warped_arr, name="warped", colormap="green", blending="additive", opacity=0.5)
viewer.add_image(np.asarray(verify_movable), name="original (hidden)", colormap="red", blending="additive", opacity=0.3, visible=False)

print("Napari viewer opened:")
print("  gray  = fixed reference")
print("  green = warped movable")
print("  red   = original movable (hidden, toggle in layer list)")

---

## 11. Nonlinear Refinement: Triangulation2D

**Use after:** Initial affine alignment to block (Mode A, B, or C)

**What it does:** Adds nonlinear warping to correct edge distortions in the slice â†’ block alignment.

**How it works:**
- Uses padding (required for point-based Triangulation2D)
- Starts from existing transform, adds nonlinear refinement

**How to use:**
1. Select an already-aligned slice
2. Run this cell
3. Press `N` (projected) or `V` (3D) when prompted
4. Click corresponding points on edges that need warping
5. Press 'q' to quit and save

In [ ]:
# =============================================================================
# NONLINEAR REFINEMENT: TRIANGULATION (requires padding)
# Same role layout as Mode A: movable = padded slice, fixed = block.
# =============================================================================

slice_name = get_selected_slice()

if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
    print("Run one of the alignment modes first.")
else:
    print(f"Refining: {slice_name}")
    print(f"Mode: Nonlinear refinement (N or V transforms)")
    print()

    # Existing transform in movable->fixed direction (slice -> block)
    existing_transform = g.get_transform(slice_name, block_node)
    print(f"Existing transform: {type(existing_transform).__name__}")

    PAD_Z = 25
    movable_img = load_and_pad_slice(slice_name, pad_z=PAD_Z)
    fixed_img   = g.get_image(block_node)

    print(f"Movable (padded {slice_name}): {movable_img.shape}")
    print(f"Fixed   ({block_node}):        {fixed_img.shape}")

    print("\n" + "="*60)
    print("LAUNCHING GUI FOR REFINEMENT")
    print("="*60)
    print("Starting from existing transform")
    print("Use 'N' for projected triangulation or 'V' for 3D triangulation")
    print("Press 'q' to quit when done")
    print()

    refined_transform = ca_gui.align_interactive(
        nodes_movable=movable_img,
        nodes_fixed=fixed_img,
        transform=existing_transform,
        graph=g,
    )

    save_alignment(slice_name, refined_transform)

    del movable_img
    gc.collect()


---

## 12. Warped Image Export

Save a warped volume as a multi-page TIFF. Set `export_target` to choose the output coordinate space:
- `block_node` â€” block space (direct transform)
- `invivo_node` â€” in-vivo space (two-hop composed automatically)

In [ ]:
# =============================================================================
# WARPED IMAGE EXPORT
# =============================================================================

# Which slice to export
slice_name = get_selected_slice()

# Choose output coordinate space
export_target = invivo_node  # or block_node for block space
export_target_img = g.get_image(export_target)

# Reuse warped_arr from verification cell if it exists
try:
    assert warped_arr is not None
    print(f"Reusing warped array from verification cell")
except (NameError, AssertionError):
    # Recompute
    print(f"Computing warped image for: {slice_name} -> {export_target}")
    t = g.get_transform(slice_name, export_target)
    slice_img = g.get_image(slice_name)
    _, warped_arr = compute_alignment_quality(export_target_img, slice_img, t)

# Export
output_path = WARPED_OUTPUT_DIR / f"{slice_name}_warped.tiff"
export_warped_image(warped_arr, output_path, dtype=np.float32)

---

## 13. Graph Reload Verification

Validates that the graph .db file correctly persists the slice â†’ block transform through SQLite serialization. Saves the graph, reloads it, re-applies the transform, and checks that the output matches.

In [ ]:
# =============================================================================
# GRAPH RELOAD VERIFICATION
# =============================================================================

slice_name = get_selected_slice()

if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
else:
    # Step 1: Get current transform output
    print(f"Verifying graph persistence for: {slice_name} -> {block_node}")
    t_original = g.get_transform(slice_name, block_node)
    slice_img = g.get_image(slice_name)
    block_img = g.get_image(block_node)
    
    warped_original = np.asarray(t_original.transform_image(
        slice_img, output_size=list(block_img.shape)
    ))
    
    # Step 2: Save graph
    g.save()
    print(f"  Graph saved to: {GRAPH_PATH}")
    
    # Step 3: Reload graph
    g_check = ca.Graph.load(str(GRAPH_PATH))
    t_reloaded = g_check.get_transform(slice_name, block_node)
    slice_img_reload = g_check.get_image(slice_name)
    
    warped_reloaded = np.asarray(t_reloaded.transform_image(
        slice_img_reload, output_size=list(block_img.shape)
    ))
    
    # Step 4: Compare
    max_diff = np.max(np.abs(warped_original.astype(np.float64) - warped_reloaded.astype(np.float64)))
    mean_diff = np.mean(np.abs(warped_original.astype(np.float64) - warped_reloaded.astype(np.float64)))
    
    print(f"\n  Original transform:  {type(t_original).__name__}")
    print(f"  Reloaded transform:  {type(t_reloaded).__name__}")
    print(f"  Max absolute diff:   {max_diff:.6e}")
    print(f"  Mean absolute diff:  {mean_diff:.6e}")
    
    if max_diff < 1e-3:
        print(f"\n  PASS: Graph persistence verified")
    else:
        print(f"\n  WARNING: Max diff {max_diff:.6e} exceeds threshold 1e-3")
        print(f"  This may indicate SQLite serialization issues (check Samba mount)")
    
    del g_check, warped_original, warped_reloaded
    gc.collect()

---

## 17. Advanced: `alignment_gui()` with Crop & References

Power-user cell showing how to call `alignment_gui()` directly instead of `align_interactive()`.

**Trade-off:** `alignment_gui()` handles a single transform at a time (no composition loop like `align_interactive()`), but supports two extra parameters:
- **`crop=True`** â€” only renders the region of the movable image that overlaps the fixed image, saving memory
- **`references`** â€” list of additional images to display as alignment aids

**Note:** `crop` does NOT work with `align_interactive()` â€” it's only available through `alignment_gui()`.

In [ ]:
# =============================================================================
# ADVANCED: alignment_gui() with crop and references
# =============================================================================
# alignment_gui() is the lower-level function that align_interactive() calls
# internally. Opens a single napari window for one transform at a time.
#
# Advantages:
#   - crop=True renders only the overlap region (lower RAM)
#   - references=[] shows additional images as guides
# Disadvantages:
#   - No composition loop -- one transform per call
#   - No save-to-graph shortcuts -- you manage saving yourself
#
# Role layout matches Mode A: movable = padded slice, fixed = block.
# =============================================================================

slice_name = get_selected_slice()

PAD_Z = 25
movable_img = load_and_pad_slice(slice_name, pad_z=PAD_Z)
fixed_img   = g.get_image(block_node)

# Starting transform (extend with point-based rigid)
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform + ca.TranslateRotate
else:
    initial_transform = get_initial_transform_slice_to_block() + ca.TranslateRotate

# Optional reference images as guides: list of (image, transform) tuples
references = []

print(f"Movable (padded {slice_name}): {movable_img.shape}")
print(f"Fixed   ({block_node}):        {fixed_img.shape}")
print("Launching alignment_gui() with crop=True (overlap-only rendering)")
print()

transform = ca_gui.alignment_gui(
    movable_image=movable_img,
    base_image=fixed_img,
    transform=initial_transform,
    crop=True,
    references=references,
)

save_alignment(slice_name, transform)

del movable_img
gc.collect()




---

## 18. Advanced: Node-Name Mode

Instead of passing raw arrays to `align_interactive()`, you can pass **node names** (strings) and let CASTalign resolve images from the graph. This is simpler but doesn't work with padding (Modes A/C) since the padded array isn't stored as a graph node.

**Use when:** Both images are already stored in the graph as nodes (e.g., Mode D, or unpadded Mode B).

In [ ]:
# =============================================================================
# ADVANCED: NODE-NAME MODE
# =============================================================================
# When both images are stored as graph nodes, pass their names (strings)
# and castalign calls g.get_image() internally.
#
# Works when:
#   - Both nodes exist in the graph
#   - You don't need padding (graph stores raw image, not padded)
#   - graph=g is passed
#
# Does NOT work for Modes A/C (they pad the slice before alignment).
#
# Role layout: movable = slice node, fixed = block node.
# =============================================================================

slice_name = get_selected_slice()

movable_node = slice_name
fixed_node   = block_node

print(f"Node-name alignment: movable={movable_node}, fixed={fixed_node}")
print(f"(Images will be loaded from graph automatically)")
print()

prev_transform = get_previous_transform(slice_name)
initial_transform = prev_transform if prev_transform is not None else get_initial_transform_slice_to_block()

transform = ca_gui.align_interactive(
    nodes_movable=movable_node,
    nodes_fixed=fixed_node,
    transform=initial_transform,
    graph=g,
)

save_alignment(slice_name, transform)


---

## 19. Troubleshooting & Tips

### Two-Hop Graph Structure
- Slices align to the ex-vivo block, and the block aligns to in-vivo.
- `g.get_transform(slice, invivo_ref_red)` automatically composes both transforms via BFS.
- The block â†’ invivo transform (Mode D) only needs to be done once.
- If you see "Path not found" errors, check that both edges exist (sliceâ†’block AND blockâ†’invivo).

### SQLite over Samba/SMB
- Graph `.db` files are SQLite databases. Writing over network mounts can cause "database is locked" or corruption errors.
- **Fix:** Copy the `.db` file locally, do alignment, copy back. Or run alignment directly on the server via SSH + X11 forwarding.
- Use the Graph Reload Verification cell (Cell 13) to confirm persistence after saving.

### Memory Management
- The block and in-vivo stacks (~200 MB each) stay in RAM once loaded by the graph. Padded slices add ~100 MB each.
- After alignment, `del padded; gc.collect()` frees the padded array.
- If napari becomes sluggish, restart the kernel and re-run from Cell 1.
- `crop=True` in `alignment_gui()` (Cell 17) reduces memory by only rendering the overlap region.

### Padding Rationale
- Point-based transforms (T, R, P, N, V) need at least a few z-slices to fit 3D correspondences.
- A 1-voxel slice has no z-extent, so the fitter can't determine z-position.
- We pad to `2*PAD_Z+1` slices (default 51) with the original at center (z=25).
- Padding value = 90% of slice mean to avoid confusing the fitter with zeros or max values.

### Label Images & `image_is_label`
- CASTalign auto-detects label images (integer dtype, few unique values) and uses nearest-neighbor interpolation.
- Cell masks and other continuous-valued images can be misdetected as labels.
- We disable this globally with `ca_utils.image_is_label = lambda img: False` in the imports cell.
- To re-enable for specific images, pass `labels=True` to `transform_image()`.

### Transform Composition & Inversion
- Transforms compose left-to-right: `t1 + t2` means "apply t1 first, then t2".
- The graph stores directed edges. `g.get_transform(A, B)` returns the A->B direction.
- CASTalign automatically inverts transforms when traversing edges in reverse.
- For two-hop: `g.get_transform(slice, invivo)` finds path sliceâ†’blockâ†’invivo and composes both transforms.

### Common Issues
- **"Point-based transforms don't fit"** â€” Make sure you have at least 3 point pairs (4+ recommended). Check that points are on the correct layers (base vs movable).
- **"Transform looks wrong after loading"** â€” Run the Graph Reload Verification cell. If it fails, the SQLite file may be corrupted (Samba issue).
- **"GUI shows blank/black"** â€” The images may have very different intensity ranges. Try normalizing before alignment, or adjust napari contrast limits manually.
- **"Path not found between nodes"** â€” The two-hop path requires both sliceâ†’block and blockâ†’invivo edges. Run Mode D if the blockâ†’invivo edge is missing.

---

## 20. MNN Alignment Validation

Quantitative alignment-quality metric built from Cellpose centroids + the fitted rigid transform. For each cell in the in-vivo volume, apply the `invivo_ref_red -> ex_vivo_block_red` transform, then mutual-nearest-neighbor (MNN) match against the ex-vivo centroids. Report the distance distribution in voxel and um.

**Prereqs:** Cellpose 3D has been run on both `BLOCK_STACK_PATH` and `INVIVO_PATH`. The `_seg.npy` files live in `<INVIVO_PATH parent>/cellpose/`.

Logic lives in `alignment/validate_mnn.py`; the cells below are thin wrappers around its functions so you can run and inspect step-by-step. Standalone CLI equivalent: `python alignment/validate_mnn.py`.


In [ ]:
# =============================================================================
# MNN VALIDATION - EXTRACT CENTROIDS
# =============================================================================
# Prereqs: cellpose _seg.npy files for both volumes, by default in
# <INVIVO_PATH parent>/cellpose/<stem>_seg.npy
# =============================================================================

# Make alignment/ importable (notebook cwd is already alignment/)
_ALIGN_DIR = Path.cwd()
if str(_ALIGN_DIR) not in sys.path:
    sys.path.insert(0, str(_ALIGN_DIR))

from validate_mnn import (
    extract_centroids,
    apply_transform_to_centroids,
    mnn_match,
    summarize,
    print_summary,
    HUANG_UM_PER_VOXEL,
    _default_seg_path,
)

from local_config import INVIVO_PATH_RED, BLOCK_STACK_PATH_RED
invivo_path = Path(INVIVO_PATH_RED)
exvivo_path = Path(BLOCK_STACK_PATH_RED)
out_dir = invivo_path.parent / "cellpose"

# Override paths here if your cellpose _seg.npy files live elsewhere.
# _default_seg_path tries next-to-TIFF first, then falls back to out_dir,
# matching where validate_mnn.main() and batch_cellpose.py write segs.
exvivo_seg = _default_seg_path(exvivo_path, out_dir)
invivo_seg = _default_seg_path(invivo_path, out_dir)

print(f"Ex-vivo seg: {exvivo_seg}  ({'OK' if exvivo_seg.exists() else 'MISSING'})")
print(f"In-vivo seg: {invivo_seg}  ({'OK' if invivo_seg.exists() else 'MISSING'})")
print()

d_ex = extract_centroids(exvivo_seg)
d_in = extract_centroids(invivo_seg)

ex_size_med = float(np.median(d_ex['sizes'])) if d_ex['sizes'].size else float('nan')
in_size_med = float(np.median(d_in['sizes'])) if d_in['sizes'].size else float('nan')
print(f"Ex-vivo: {len(d_ex['centroids']):>6d} cells, median size {ex_size_med:.0f} voxels")
print(f"In-vivo: {len(d_in['centroids']):>6d} cells, median size {in_size_med:.0f} voxels")


In [ ]:
# =============================================================================
# MNN VALIDATION - APPLY RIGID TRANSFORM
# =============================================================================
# Queries the in-vivo -> ex-vivo-block edge (auto-composed for multi-hop paths).
# Centroids move from in-vivo voxel frame into ex-vivo voxel frame.
# =============================================================================

transform = g.get_transform(invivo_node, block_node)
print(f"Transform: {type(transform).__name__}")

in_in_ex_frame = apply_transform_to_centroids(transform, d_in['centroids'])

def _bbox_line(pts, label):
    if pts.shape[0] == 0:
        return f"{label}: (empty)"
    mn, mx = pts.min(axis=0), pts.max(axis=0)
    return (f"{label}: z {mn[0]:7.1f}-{mx[0]:7.1f}  "
            f"y {mn[1]:7.1f}-{mx[1]:7.1f}  x {mn[2]:7.1f}-{mx[2]:7.1f}")

print()
print(_bbox_line(d_ex['centroids'],  "ex-vivo             "))
print(_bbox_line(d_in['centroids'],  "in-vivo (raw)       "))
print(_bbox_line(in_in_ex_frame,     "in-vivo (ex-frame)  "))


In [ ]:
# =============================================================================
# MNN VALIDATION - MUTUAL NEAREST NEIGHBOR MATCH + SUMMARY
# =============================================================================
# KDTree bidirectional NN in voxel space. Distances reported in voxel and um
# (per-axis, since Z pitch is ~1.8x XY pitch).
# =============================================================================

mnn = mnn_match(in_in_ex_frame, d_ex['centroids'])
summary = summarize(
    mnn,
    n_movable=len(d_in['centroids']),
    n_fixed=len(d_ex['centroids']),
)
print_summary(summary)


In [ ]:
# =============================================================================
# MNN VALIDATION - NAPARI OVERLAY
# =============================================================================
# Plots both centroid sets on the ex-vivo volume in the ex-vivo frame.
# Matched points -> green, unmatched -> grey. Visual QC of MNN matches.
# =============================================================================
import napari

ex_colors = np.tile([0.5, 0.5, 0.5, 0.6], (len(d_ex['centroids']), 1))
in_colors = np.tile([0.5, 0.5, 0.5, 0.6], (len(in_in_ex_frame), 1))
if mnn['fixed_idx'].size:
    ex_colors[mnn['fixed_idx']] = [0.0, 1.0, 0.0, 1.0]
if mnn['movable_idx'].size:
    in_colors[mnn['movable_idx']] = [0.0, 1.0, 0.0, 1.0]

viewer = napari.Viewer(title="MNN validation (ex-vivo frame)")
viewer.add_image(
    np.asarray(g.get_image(block_node)),
    name="ex_vivo_block_red", colormap="gray", blending="additive", opacity=0.7,
)
viewer.add_points(
    d_ex['centroids'], name="ex-vivo centroids",
    face_color=ex_colors, size=4, symbol='disc',
)
viewer.add_points(
    in_in_ex_frame, name="in-vivo centroids (transformed)",
    face_color=in_colors, size=4, symbol='cross',
)

print("Napari viewer opened:")
print("  disc  = ex-vivo centroids")
print("  cross = in-vivo centroids transformed into ex-vivo frame")
print("  green = MNN-matched pair, grey = unmatched")


In [ ]:
# =============================================================================
# MNN VALIDATION - DISTANCE HISTOGRAM
# =============================================================================
# Saves mnn_histogram.png next to the other cellpose-derived outputs.
# =============================================================================
import matplotlib.pyplot as plt

d_um = np.linalg.norm(mnn['distances_um_per_axis'], axis=1)
d_vox = mnn['distances_voxel']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, unit in [(axes[0], d_vox, 'voxel'), (axes[1], d_um, 'um')]:
    if data.size == 0:
        ax.text(0.5, 0.5, "no MNN matches", ha='center', va='center', transform=ax.transAxes)
        ax.set_xlabel(f"MNN distance ({unit})")
        continue
    ax.hist(data, bins=40, color='steelblue', alpha=0.85)
    med = float(np.median(data))
    p95 = float(np.percentile(data, 95))
    ax.axvline(med, color='red',    linestyle='--', label=f"median {med:.2f}")
    ax.axvline(p95, color='orange', linestyle='--', label=f"p95 {p95:.2f}")
    ax.set_xlabel(f"MNN distance ({unit})")
    ax.set_ylabel("count")
    ax.legend()

plt.tight_layout()
out_hist = out_dir / "mnn_histogram.png"
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_hist, dpi=120)
print(f"Saved: {out_hist}")
plt.show()
